# LeetCode 1237 — Find Positive Integer Solution for a Given Equation bằng Python

Link đề: https://leetcode.com/problems/find-positive-integer-solution-for-a-given-equation/ · Độ khó: **Medium** · Chủ đề: Math, Two Pointers, Binary Search, Interactive

Skeleton LeetCode:

```python
"""
   This is the custom function interface.
   You should not implement it, or speculate about its implementation
   class CustomFunction:
       # Returns f(x, y) for any given positive integers x and y.
       # Note that f(x, y) is increasing with respect to both x and y.
       # i.e. f(x, y) < f(x + 1, y), f(x, y) < f(x, y + 1)
       def f(self, x, y):

"""

class Solution:
    def findSolution(self, customfunction: 'CustomFunction', z: int) -> List[List[int]]:
        ...
```

Ghi chú: bài **Interactive** — ta không biết công thức của `f`, chỉ được gọi `customfunction.f(x, y)`. Điều duy nhất ta biết: `f` **tăng ngặt** theo từng biến. Kết quả trả về theo thứ tự bất kỳ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu khái niệm **hàm đơn điệu tăng ngặt hai biến** (strictly increasing) và cách khai thác nó mà không cần biết công thức.
2. Viết được lời giải vét cạn có **cắt nhánh** (pruning) và đếm số lần gọi `f`.
3. Áp dụng **tìm kiếm nhị phân** (binary search) theo `y` cho từng `x` cố định.
4. Nắm kỹ thuật **hai con trỏ trên "ma trận sắp xếp"** (giống bài Search a 2D Matrix II): `x` tăng, `y` giảm, mỗi bước loại bỏ một hàng hoặc một cột.
5. Rút ra chặn trên tinh tế `x, y ≤ z` từ tính tăng ngặt trên số nguyên, giúp thu nhỏ không gian tìm kiếm.

## 2. Đề bài & ví dụ

Cho một hàm `f(x, y)` **ẩn công thức** và một giá trị `z`. Trả về **mọi cặp số nguyên dương** `[x, y]` sao cho `f(x, y) == z`, theo thứ tự bất kỳ.

Dù công thức bị giấu, ta biết `f` **tăng đơn điệu ngặt**:

- `f(x, y) < f(x + 1, y)`
- `f(x, y) < f(x, y + 1)`

Trình chấm có 9 cài đặt ẩn của `CustomFunction`; nó nhận `function_id` và `z`, gọi `findSolution` của bạn rồi so với đáp án.

**Ví dụ 1:** `function_id = 1` (`f(x, y) = x + y`), `z = 5` → `[[1,4],[2,3],[3,2],[4,1]]`.

**Ví dụ 2:** `function_id = 2` (`f(x, y) = x * y`), `z = 5` → `[[1,5],[5,1]]`.

**Ràng buộc:**

- `1 <= function_id <= 9`, `1 <= z <= 100`.
- Đảm bảo mọi nghiệm của `f(x, y) == z` nằm trong `1 <= x, y <= 1000`.
- `f(x, y)` vừa số nguyên 32 bit có dấu khi `1 <= x, y <= 1000`.

Để chạy offline, ta tự dựng `CustomFunction` giả lập. Lớp này nhận một hàm Python bất kỳ (miễn là tăng ngặt theo cả hai biến) và **đếm số lần gọi** `f`. Các công thức dưới đây chỉ là ví dụ minh hoạ, không phải 9 công thức thật của trình chấm.

In [1]:
import bisect
import random
import time
from typing import List


class CustomFunction:
    def __init__(self, func):
        self.func = func
        self.calls = 0

    def f(self, x, y):
        self.calls += 1
        return self.func(x, y)


FUNCS = {
    1: lambda x, y: x + y,
    2: lambda x, y: x * y,
    3: lambda x, y: x * x + y,
    4: lambda x, y: x + y * y,
    5: lambda x, y: x * x + y * y,
    6: lambda x, y: (x + y) * (x + y),
    7: lambda x, y: x ** 3 + y ** 3,
    8: lambda x, y: x * x * y,
    9: lambda x, y: x * y * y,
}

examples = [(1, 5, [[1, 4], [2, 3], [3, 2], [4, 1]]), (2, 5, [[1, 5], [5, 1]])]
cf = CustomFunction(FUNCS[1])
print(cf.f(2, 3), cf.f(4, 1), "| số lần gọi:", cf.calls)

5 5 | số lần gọi: 2


## 3. Kiến thức nền

### 3.1 Hàm tăng ngặt hai biến

"Tăng ngặt theo `x`" nghĩa là giữ nguyên `y`, tăng `x` thì `f` **lớn hơn hẳn** (không bằng). Tương tự theo `y`. Hệ quả: nếu `x1 ≤ x2` và `y1 ≤ y2` (không cùng bằng) thì `f(x1, y1) < f(x2, y2)`.

Ta kiểm tra tính chất này cho 9 hàm minh hoạ trên lưới nhỏ.

In [2]:
def is_strictly_increasing(func, n=30):
    for x in range(1, n + 1):
        for y in range(1, n + 1):
            if not (func(x, y) < func(x + 1, y) and func(x, y) < func(x, y + 1)):
                return False
    return True


print({k: is_strictly_increasing(g) for k, g in FUNCS.items()})
print("x - y có tăng ngặt không?", is_strictly_increasing(lambda x, y: x - y))

{1: True, 2: True, 3: True, 4: True, 5: True, 6: True, 7: True, 8: True, 9: True}
x - y có tăng ngặt không? False


### 3.2 Nhìn `f` như một ma trận đã sắp xếp

Viết bảng `M[x][y] = f(x, y)`. Mỗi **hàng** tăng dần từ trái sang phải (theo `y`), mỗi **cột** tăng dần từ trên xuống (theo `x`). Đây chính là "ma trận sắp xếp theo hàng và cột" — cấu trúc nổi tiếng trong bài *Search a 2D Matrix II* (LeetCode 240).

In [3]:
g = FUNCS[1]
print("     " + " ".join(f"y={y:<2}" for y in range(1, 7)))
for x in range(1, 7):
    print(f"x={x}: " + " ".join(f"{g(x, y):4}" for y in range(1, 7)))

     y=1  y=2  y=3  y=4  y=5  y=6 
x=1:    2    3    4    5    6    7
x=2:    3    4    5    6    7    8
x=3:    4    5    6    7    8    9
x=4:    5    6    7    8    9   10
x=5:    6    7    8    9   10   11
x=6:    7    8    9   10   11   12


### 3.3 Tìm kiếm nhị phân trên một dãy tăng

Với `x` cố định, dãy `f(x, 1), f(x, 2), …` tăng ngặt, nên tìm `y` có `f(x, y) == z` bằng nhị phân: so sánh giá trị giữa với `z` rồi bỏ một nửa. Mỗi lần tìm tốn `O(log Y)` lời gọi.

In [4]:
def bin_search_y(cf, x, z, lo=1, hi=1000):
    while lo <= hi:
        mid = (lo + hi) // 2
        v = cf.f(x, mid)
        if v == z:
            return mid
        if v < z:
            lo = mid + 1
        else:
            hi = mid - 1
    return None


cf = CustomFunction(FUNCS[2])
print("x=1:", bin_search_y(cf, 1, 5), "| x=2:", bin_search_y(cf, 2, 5), "| số lần gọi:", cf.calls)

x=1: 5 | x=2: None | số lần gọi: 19


### 3.4 Thư viện `bisect` (tham khảo)

`bisect.bisect_left(a, t)` trả về vị trí đầu tiên có `a[i] >= t` trong list đã sắp xếp. Từ Python 3.10 có tham số `key`, nhưng nó cần một **dãy** thật; với hàm ẩn ta có thể bọc trong một đối tượng có `__getitem__` và `__len__`. Ở đây ta minh hoạ trên list thường.

In [5]:
a = [3, 6, 9, 12, 15]
print(bisect.bisect_left(a, 9), bisect.bisect_left(a, 10), bisect.bisect_left(a, 100))


class Row:
    """Cho phép bisect trên hàng x của f mà không tạo list."""

    def __init__(self, cf, x, n):
        self.cf, self.x, self.n = cf, x, n

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        return self.cf.f(self.x, i + 1)


cf = CustomFunction(FUNCS[1])
i = bisect.bisect_left(Row(cf, 2, 1000), 5)
print("x=2, y =", i + 1, "f =", FUNCS[1](2, i + 1), "| số lần gọi:", cf.calls)

2 3 5
x=2, y = 3 f = 5 | số lần gọi: 10


## 4. Ý tưởng nền tảng / trực giác

Tính tay với `f = x + y`, `z = 5`. Xếp bảng nhỏ và tô các ô bằng 5:

| x \ y | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| 1 | 2 | 3 | 4 | **5** | 6 |
| 2 | 3 | 4 | **5** | 6 | 7 |
| 3 | 4 | **5** | 6 | 7 | 8 |
| 4 | **5** | 6 | 7 | 8 | 9 |
| 5 | 6 | 7 | 8 | 9 | 10 |

Quan sát: các ô bằng `z` tạo thành một "đường chéo đi xuống bên trái". Khi `x` tăng thì `y` nghiệm **giảm** — vì nếu `f(x, y) = z` và `x' > x` thì `f(x', y) > z`, buộc `y' < y`. Do đó **mỗi hàng có tối đa một nghiệm** và các nghiệm sắp theo `x` tăng thì `y` giảm.

In [6]:
z = 5
g = FUNCS[1]
sols = [(x, y) for x in range(1, 6) for y in range(1, 6) if g(x, y) == z]
print(sols)
ys = [y for _, y in sols]
assert ys == sorted(ys, reverse=True)

[(1, 4), (2, 3), (3, 2), (4, 1)]


## 5. Cách 1 — Brute Force / baseline

Duyệt mọi `x` từ 1 tới 1000 và mọi `y` từ 1 tới 1000, giữ cặp có `f(x, y) == z`. Tốn đúng **10⁶ lời gọi** — quá chậm khi chạy nhiều test trong notebook, nên ta thêm một cắt nhánh tự nhiên: với `x` cố định, hễ `f(x, y) > z` thì các `y` lớn hơn chỉ càng lớn → `break`. Ta để giới hạn `LIM` làm tham số để còn kiểm thử.

In [7]:
class SolutionBrute:
    def findSolution(self, customfunction, z, LIM=1000):
        ans = []
        for x in range(1, LIM + 1):
            for y in range(1, LIM + 1):
                v = customfunction.f(x, y)
                if v == z:
                    ans.append([x, y])
                if v >= z:          # cắt nhánh: y lớn hơn chỉ cho f lớn hơn
                    break
        return ans


for fid, z, want in examples:
    cf = CustomFunction(FUNCS[fid])
    got = SolutionBrute().findSolution(cf, z)
    print(sorted(got) == sorted(want), got, "| lời gọi:", cf.calls)

True [[1, 4], [2, 3], [3, 2], [4, 1]] | lời gọi: 1006
True [[1, 5], [5, 1]] | lời gọi: 1008


**Độ phức tạp:** không cắt nhánh: `O(X · Y) = 10⁶` lời gọi. Có cắt nhánh: mỗi hàng `x` vẫn tốn ít nhất 1 lời gọi, nên tổng `O(X + số ô < z)`; với `f = x + y` thì khoảng `X + z²/2`. Nhưng **mọi `x` từ 1 tới 1000 đều bị thử** dù chỉ vài `x` đầu có thể có nghiệm.

**Nút thắt:** ta dùng tính đơn điệu theo `y` nhưng **bỏ phí** tính đơn điệu theo `x` và mối liên hệ giữa các hàng.

## 6. Bước đột phá

**Insight 1 (hai con trỏ):** đặt `x = 1` (hàng nhỏ nhất) và `y = Y` (cột lớn nhất) — góc **trên phải** của bảng. Từ góc này:

- Nếu `f(x, y) < z` thì **mọi** ô cùng hàng `x` bên trái còn nhỏ hơn → hàng `x` hết hi vọng → `x += 1`.
- Nếu `f(x, y) > z` thì **mọi** ô cùng cột `y` bên dưới còn lớn hơn → cột `y` hết hi vọng → `y -= 1`.
- Nếu `f(x, y) == z` thì ghi nhận; hàng `x` và cột `y` không còn nghiệm khác (mỗi hàng/cột tối đa một nghiệm) → `x += 1`, `y -= 1`.

Mỗi bước loại bỏ ít nhất một hàng hoặc một cột → tối đa `X + Y` bước.

**Invariant:** mọi nghiệm chưa được ghi nhận đều nằm trong hình chữ nhật `[x..X] × [1..y]`.

**Insight 2 (chặn trên bằng `z`):** vì `f` nhận giá trị nguyên dương và tăng ngặt, mỗi lần tăng một biến lên 1 thì `f` tăng ít nhất 1. Do đó `f(x, y) ≥ f(1, 1) + (x - 1) + (y - 1) ≥ x + y - 1`. Nếu `f(x, y) = z` thì `x + y - 1 ≤ z`, suy ra `x ≤ z` và `y ≤ z` (vì biến kia ≥ 1). Với `z ≤ 100` ta chỉ cần xét `x, y ∈ [1, z]` thay vì `[1, 1000]`.

In [8]:
# Kiểm chứng insight 2: f(x, y) >= x + y - 1 cho các hàm minh hoạ
ok = all(g(x, y) >= x + y - 1 for g in FUNCS.values() for x in range(1, 60) for y in range(1, 60))
print("f(x,y) >= x+y-1 trên 9 hàm minh hoạ:", ok)

f(x,y) >= x+y-1 trên 9 hàm minh hoạ: True


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Nhị phân theo `y` cho từng `x`

Với mỗi `x`, tìm nhị phân `y ∈ [1, 1000]`. Thêm cắt nhánh: nếu `f(x, 1) > z` thì các `x` lớn hơn cũng vô vọng → dừng hẳn.

In [9]:
class SolutionBinary:
    def findSolution(self, customfunction, z):
        ans = []
        for x in range(1, 1001):
            if customfunction.f(x, 1) > z:     # hàng x và các hàng sau đều > z
                break
            lo, hi = 1, 1000
            while lo <= hi:
                mid = (lo + hi) // 2
                v = customfunction.f(x, mid)
                if v == z:
                    ans.append([x, mid])
                    break
                if v < z:
                    lo = mid + 1
                else:
                    hi = mid - 1
        return ans


for fid, z, want in examples:
    cf = CustomFunction(FUNCS[fid])
    got = SolutionBinary().findSolution(cf, z)
    print(sorted(got) == sorted(want), got, "| lời gọi:", cf.calls)

True [[1, 4], [2, 3], [3, 2], [4, 1]] | lời gọi: 42
True [[1, 5], [5, 1]] | lời gọi: 54


**Độ phức tạp:** `O(X · log Y)` lời gọi ≈ `1000 · 10 = 10⁴` trong trường hợp xấu (không có cắt nhánh). Bộ nhớ `O(1)` ngoài đáp án.

### 7.2 Cách 3 — Hai con trỏ (góc trên phải)

Đây là lời giải kinh điển: `x` bắt đầu từ 1, `y` bắt đầu từ 1000.

In [10]:
class SolutionTwoPointers:
    def findSolution(self, customfunction, z):
        ans = []
        x, y = 1, 1000
        while x <= 1000 and y >= 1:
            v = customfunction.f(x, y)
            if v < z:
                x += 1
            elif v > z:
                y -= 1
            else:
                ans.append([x, y])
                x += 1
                y -= 1
        return ans


for fid, z, want in examples:
    cf = CustomFunction(FUNCS[fid])
    got = SolutionTwoPointers().findSolution(cf, z)
    print(sorted(got) == sorted(want), got, "| lời gọi:", cf.calls)

True [[1, 4], [2, 3], [3, 2], [4, 1]] | lời gọi: 1000
True [[1, 5], [5, 1]] | lời gọi: 1003


**Độ phức tạp:** mỗi vòng `x` tăng hoặc `y` giảm (hoặc cả hai) → tối đa `X + Y - 1 ≈ 2000` lời gọi. Bộ nhớ `O(1)`.

### 7.3 Cách 4 — Hai con trỏ với chặn trên `z`

Kết hợp Insight 2: chỉ cần `x, y ∈ [1, z]`. Bắt đầu `y = z`. Số lời gọi giảm còn `≤ 2z ≤ 200`.

In [11]:
class SolutionTwoPointersBounded:
    def findSolution(self, customfunction, z):
        ans = []
        x, y = 1, z
        while x <= z and y >= 1:
            v = customfunction.f(x, y)
            if v < z:
                x += 1
            elif v > z:
                y -= 1
            else:
                ans.append([x, y])
                x += 1
                y -= 1
        return ans


for fid, z, want in examples:
    cf = CustomFunction(FUNCS[fid])
    got = SolutionTwoPointersBounded().findSolution(cf, z)
    print(sorted(got) == sorted(want), got, "| lời gọi:", cf.calls)

True [[1, 4], [2, 3], [3, 2], [4, 1]] | lời gọi: 5
True [[1, 5], [5, 1]] | lời gọi: 8


### 7.4 Cách 5 — Nhị phân với cận co dần

Dùng nhị phân như Cách 2 nhưng cận trên của `y` **co lại** sau mỗi hàng: nếu ở hàng `x` ta biết mọi `y > hi` cho `f > z`, thì ở hàng `x + 1` điều đó vẫn đúng (cột tăng xuống dưới). Thêm chặn `z`.

In [12]:
class SolutionBinaryShrink:
    def findSolution(self, customfunction, z):
        ans = []
        hi_y = z
        for x in range(1, z + 1):
            # tìm y lớn nhất trong [1, hi_y] có f(x, y) <= z
            lo, hi = 1, hi_y
            best = 0
            while lo <= hi:
                mid = (lo + hi) // 2
                if customfunction.f(x, mid) <= z:
                    best = mid
                    lo = mid + 1
                else:
                    hi = mid - 1
            if best == 0:          # ngay f(x, 1) > z -> các hàng sau cũng vậy
                break
            if customfunction.f(x, best) == z:
                ans.append([x, best])
            hi_y = best            # hàng sau: y > best chắc chắn cho f > z
        return ans


for fid, z, want in examples:
    cf = CustomFunction(FUNCS[fid])
    got = SolutionBinaryShrink().findSolution(cf, z)
    print(sorted(got) == sorted(want), got, "| lời gọi:", cf.calls)

True [[1, 4], [2, 3], [3, 2], [4, 1]] | lời gọi: 15
True [[1, 5], [5, 1]] | lời gọi: 15


**Độ phức tạp:** `O(z · log z)` lời gọi trong trường hợp xấu; thường ít hơn nhờ cận co dần. Cách này hữu ích khi một bên rất lớn và bên kia nhỏ.

Lưu ý: cả Cách 4 và 5 dựa vào việc `f` trả **số nguyên dương** — đúng với đề. Nếu `f` trả số thực thì Insight 2 không còn đúng.

## 8. TRACE từng bước

Trace hai con trỏ (Cách 4) với `f = x * y`, `z = 5`, bắt đầu `x = 1, y = 5`:

| Bước | x | y | f(x, y) | So với z | Hành động |
|---|---|---|---|---|---|
| 1 | 1 | 5 | 5 | = | ghi [1,5], x=2, y=4 |
| 2 | 2 | 4 | 8 | > | y=3 |
| 3 | 2 | 3 | 6 | > | y=2 |
| 4 | 2 | 2 | 4 | < | x=3 |
| 5 | 3 | 2 | 6 | > | y=1 |
| 6 | 3 | 1 | 3 | < | x=4 |
| 7 | 4 | 1 | 4 | < | x=5 |
| 8 | 5 | 1 | 5 | = | ghi [5,1], x=6, y=0 → dừng |

In [13]:
def trace_two_pointers(func, z):
    cf = CustomFunction(func)
    ans = []
    x, y = 1, z
    step = 0
    while x <= z and y >= 1:
        step += 1
        v = cf.f(x, y)
        if v < z:
            act = f"< z -> x={x + 1}"
            x += 1
        elif v > z:
            act = f"> z -> y={y - 1}"
            y -= 1
        else:
            ans.append([x, y])
            act = f"= z -> ghi [{x},{y}], x={x + 1}, y={y - 1}"
            x += 1
            y -= 1
        print(f"bước {step}: f={v:3} {act}")
    print("kết quả:", ans, "| lời gọi:", cf.calls)
    return ans


trace_two_pointers(FUNCS[2], 5)

bước 1: f=  5 = z -> ghi [1,5], x=2, y=4
bước 2: f=  8 > z -> y=3
bước 3: f=  6 > z -> y=2
bước 4: f=  4 < z -> x=3
bước 5: f=  6 > z -> y=1
bước 6: f=  3 < z -> x=4
bước 7: f=  4 < z -> x=5
bước 8: f=  5 = z -> ghi [5,1], x=6, y=0
kết quả: [[1, 5], [5, 1]] | lời gọi: 8


[[1, 5], [5, 1]]

Trace thêm với `f = x + y`, `z = 5`: con trỏ đi chéo, cứ mỗi bước gặp đúng một nghiệm.

In [14]:
trace_two_pointers(FUNCS[1], 5)

bước 1: f=  6 > z -> y=4
bước 2: f=  5 = z -> ghi [1,4], x=2, y=3
bước 3: f=  5 = z -> ghi [2,3], x=3, y=2
bước 4: f=  5 = z -> ghi [3,2], x=4, y=1
bước 5: f=  5 = z -> ghi [4,1], x=5, y=0
kết quả: [[1, 4], [2, 3], [3, 2], [4, 1]] | lời gọi: 5


[[1, 4], [2, 3], [3, 2], [4, 1]]

## 9. Code nộp LeetCode

In [15]:
"""
   This is the custom function interface.
   You should not implement it, or speculate about its implementation
   class CustomFunction:
       # Returns f(x, y) for any given positive integers x and y.
       # Note that f(x, y) is increasing with respect to both x and y.
       # i.e. f(x, y) < f(x + 1, y), f(x, y) < f(x, y + 1)
       def f(self, x, y):

"""
from typing import List


class Solution:
    def findSolution(self, customfunction: 'CustomFunction', z: int) -> List[List[int]]:
        ans = []
        x, y = 1, min(z, 1000)
        while x <= 1000 and y >= 1:
            v = customfunction.f(x, y)
            if v < z:
                x += 1
            elif v > z:
                y -= 1
            else:
                ans.append([x, y])
                x += 1
                y -= 1
        return ans


for fid, z, want in examples:
    assert sorted(Solution().findSolution(CustomFunction(FUNCS[fid]), z)) == sorted(want)
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


Ghi chú: ta bắt đầu `y = min(z, 1000)` (dùng Insight 2) nhưng giữ điều kiện `x <= 1000` an toàn; vòng lặp vẫn dừng sớm vì `y` giảm hoặc `x` tăng mỗi bước, và khi `x > z` thì `f(x, y) > z` nên `y` sẽ giảm nhanh về 0.

## 10. Vì sao đúng

Xét Cách 3/4 với hình chữ nhật còn lại `[x..X] × [1..y]`.

**Invariant:** mọi nghiệm **chưa ghi nhận** đều nằm trong `[x..X] × [1..y]`.

- Ban đầu `x = 1`, `y = Y`: toàn bộ không gian → đúng (với Cách 4, Insight 2 chứng minh không có nghiệm ngoài `[1..z]²`).
- Nếu `f(x, y) < z`: với mọi `y' ≤ y`, `f(x, y') ≤ f(x, y) < z` → hàng `x` không có nghiệm trong phần còn lại → bỏ hàng `x` (`x += 1`) giữ invariant.
- Nếu `f(x, y) > z`: với mọi `x' ≥ x`, `f(x', y) ≥ f(x, y) > z` → cột `y` không có nghiệm → bỏ cột (`y -= 1`).
- Nếu `f(x, y) = z`: các ô khác trong hàng `x` có `y' < y` cho `f < z`; các ô khác trong cột `y` có `x' > x` cho `f > z` → bỏ cả hàng và cột sau khi ghi nhận.

Khi vòng lặp dừng, hình chữ nhật rỗng → không còn nghiệm nào chưa ghi. Mỗi nghiệm được ghi đúng một lần vì hàng của nó bị loại ngay sau đó. **Dừng:** mỗi bước giảm `(X - x) + y` ít nhất 1.

## 11. Time / Space Complexity

Đặt `X = Y = 1000` (miền tìm kiếm của đề), `z ≤ 100`. Đơn vị là **số lần gọi `f`** (mỗi lần coi như O(1)).

| Cách | Lời gọi (worst) | Space phụ |
|---|---|---|
| 1. Vét cạn (không cắt) | `X · Y = 10⁶` | `O(1)` |
| 1'. Vét cạn có cắt theo `y` | `O(X + z²)` | `O(1)` |
| 2. Nhị phân theo `y` | `O(X · log Y) ≈ 10⁴` | `O(1)` |
| 3. Hai con trỏ | `≤ X + Y ≈ 2000` | `O(1)` |
| 4. Hai con trỏ + chặn `z` | `≤ 2z ≤ 200` | `O(1)` |
| 5. Nhị phân cận co dần + chặn `z` | `O(z log z)` | `O(1)` |

Đáp án có tối đa `min(X, Y)` cặp (mỗi hàng ≤ 1 nghiệm), chiếm `O(số nghiệm)` bộ nhớ.

## 12. Bộ test

### 12.1 Ví dụ đề và edge cases

In [16]:
ALL = [SolutionBinary, SolutionTwoPointers, SolutionTwoPointersBounded, SolutionBinaryShrink, Solution]


def norm(ans):
    return sorted(map(tuple, ans))


def run_all(func, z):
    base = norm(SolutionBrute().findSolution(CustomFunction(func), z, LIM=120))
    for cls in ALL:
        got = norm(cls().findSolution(CustomFunction(func), z))
        assert got == base, (cls.__name__, got, base)
    return base


for fid, z, want in examples:
    assert run_all(FUNCS[fid], z) == norm(want)

# z = 1: x + y không thể bằng 1 (x, y >= 1)
assert run_all(FUNCS[1], 1) == []
# z = 2 với x + y -> chỉ (1, 1)
assert run_all(FUNCS[1], 2) == [(1, 1)]
# x*y với z = 100: mọi ước
assert run_all(FUNCS[2], 100) == [(d, 100 // d) for d in range(1, 101) if 100 % d == 0]
# z không đạt được: x^2 + y^2 = 3
assert run_all(FUNCS[5], 3) == []
print("edge cases OK")

edge cases OK


### 12.2 Ngẫu nhiên có seed: hàm đơn điệu ngẫu nhiên

Sinh hàm tăng ngặt ngẫu nhiên dạng `f(x, y) = a·x^p + b·y^q + c·x·y + d` với hệ số không âm, `a, b ≥ 1`. So mọi cách với vét cạn.

In [17]:
def random_monotone(rng):
    a, b = rng.randint(1, 3), rng.randint(1, 3)
    c, d = rng.randint(0, 2), rng.randint(0, 5)
    p, q = rng.randint(1, 3), rng.randint(1, 3)
    return lambda x, y: a * x ** p + b * y ** q + c * x * y + d


rng = random.Random(1237)
cnt_nonempty = 0
for _ in range(300):
    func = random_monotone(rng)
    z = rng.randint(1, 100)
    res = run_all(func, z)
    cnt_nonempty += bool(res)
for fid, func in FUNCS.items():
    for z in range(1, 101):
        run_all(func, z)
print("300 hàm ngẫu nhiên + 9 hàm x 100 giá trị z OK; số test có nghiệm:", cnt_nonempty)

300 hàm ngẫu nhiên + 9 hàm x 100 giá trị z OK; số test có nghiệm: 111


### 12.3 Đếm số lời gọi — test "giới hạn"

So số lần gọi `f` của từng cách ở `z = 100` với `f = x + y` (nhiều nghiệm nhất) và `f = x * y`.

In [18]:
for fid in (1, 2):
    row = []
    for cls in [SolutionBinary, SolutionTwoPointers, SolutionTwoPointersBounded, SolutionBinaryShrink, Solution]:
        cf = CustomFunction(FUNCS[fid])
        cls().findSolution(cf, 100)
        row.append(f"{cls.__name__}={cf.calls}")
    cf = CustomFunction(FUNCS[fid])
    SolutionBrute().findSolution(cf, 100)
    row.append(f"Brute(cắt)={cf.calls}")
    print(f"f{fid}:", ", ".join(row))

t = time.perf_counter()
for _ in range(200):
    Solution().findSolution(CustomFunction(FUNCS[7]), 100)
print(f"200 lần Solution: {(time.perf_counter() - t) * 1000:.1f} ms")

f1: SolutionBinary=992, SolutionTwoPointers=1000, SolutionTwoPointersBounded=100, SolutionBinaryShrink=679, Solution=100, Brute(cắt)=5851
f2: SolutionBinary=1096, SolutionTwoPointers=1091, SolutionTwoPointersBounded=191, SolutionBinaryShrink=300, Solution=191, Brute(cắt)=1473
200 lần Solution: 5.0 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bắt đầu hai con trỏ ở góc sai (`x = 1, y = 1`)

Ở góc trên trái, cả hai hướng đi đều làm `f` **tăng** — không có cách quyết định nên bỏ hàng hay bỏ cột.

In [19]:
def two_pointers_wrong_corner(cf, z):
    ans = []
    x, y = 1, 1
    while x <= z and y <= z:
        v = cf.f(x, y)
        if v == z:
            ans.append([x, y])
            x += 1
        elif v < z:
            y += 1      # đoán bừa: tăng y
        else:
            x += 1
    return ans


print("sai :", two_pointers_wrong_corner(CustomFunction(FUNCS[1]), 5))
print("đúng:", Solution().findSolution(CustomFunction(FUNCS[1]), 5))

sai : [[1, 4]]
đúng: [[1, 4], [2, 3], [3, 2], [4, 1]]


**Vì sao sai:** sau khi tìm `[1,4]`, `x = 2, y = 4` → `f = 6 > 5` → `x` tăng mãi, bỏ sót `[2,3],[3,2],[4,1]` (cần **giảm** `y` mà code không có nước đi đó). **Sửa:** bắt đầu từ góc mà hai hướng đi có tác dụng **ngược nhau**: `(x nhỏ, y lớn)` hoặc `(x lớn, y nhỏ)`.

### Lỗi 2 — Gặp nghiệm rồi chỉ tăng `x` mà quên giảm `y` (hoặc ngược lại) với vòng lặp sai điều kiện

Thực ra chỉ tăng `x` vẫn đúng (bước sau `f > z` sẽ giảm `y`), nhưng nếu gặp nghiệm mà **không di chuyển gì** thì lặp vô hạn. Ta minh hoạ phiên bản giới hạn số bước để thấy nó kẹt.

In [20]:
def two_pointers_stuck(cf, z, max_steps=50):
    ans = []
    x, y = 1, z
    steps = 0
    while x <= z and y >= 1 and steps < max_steps:
        steps += 1
        v = cf.f(x, y)
        if v < z:
            x += 1
        elif v > z:
            y -= 1
        else:
            ans.append([x, y])          # SAI: quên di chuyển con trỏ
    return ans, steps


ans, steps = two_pointers_stuck(CustomFunction(FUNCS[1]), 5)
print("số bước:", steps, "| số cặp ghi (trùng):", len(ans), ans[:3])

số bước: 50 | số cặp ghi (trùng): 49 [[1, 4], [1, 4], [1, 4]]


**Vì sao sai:** vòng lặp đứng yên ở `(1, 4)` và ghi trùng mãi (ở đây bị chặn bởi `max_steps`, trên LeetCode sẽ TLE/MLE). **Sửa:** khi `f == z` thì `x += 1` và `y -= 1`.

### Lỗi 3 — Nhị phân không cắt nhánh và giới hạn sai (`hi = z` với cách không dùng số nguyên)

Một lỗi khác: dùng `range(1, z)` (thiếu `z`) khi áp dụng chặn trên.

In [21]:
def two_pointers_off_by_one(cf, z):
    ans = []
    x, y = 1, z - 1                      # SAI: bỏ mất y = z
    while x < z and y >= 1:              # SAI: bỏ mất x = z
        v = cf.f(x, y)
        if v < z:
            x += 1
        elif v > z:
            y -= 1
        else:
            ans.append([x, y])
            x += 1
            y -= 1
    return ans


print("sai :", two_pointers_off_by_one(CustomFunction(FUNCS[2]), 5))
print("đúng:", Solution().findSolution(CustomFunction(FUNCS[2]), 5))

sai : []
đúng: [[1, 5], [5, 1]]


**Vì sao sai:** với `f = x·y, z = 5`, nghiệm `[1,5]` có `y = z` và `[5,1]` có `x = z` — đúng sát chặn trên. **Sửa:** chặn là `x ≤ z`, `y ≤ z` (bao gồm `z`).

### Lỗi 4 — Tưởng nghiệm đối xứng (`[x,y]` thì có `[y,x]`)

In [22]:
def solve_assume_symmetric(cf, z):
    ans = set()
    for x in range(1, z + 1):
        for y in range(x, z + 1):         # chỉ xét y >= x rồi "lật"
            if cf.f(x, y) == z:
                ans.add((x, y))
                ans.add((y, x))
    return sorted(ans)


print("sai :", solve_assume_symmetric(CustomFunction(FUNCS[3]), 5))
print("đúng:", norm(Solution().findSolution(CustomFunction(FUNCS[3]), 5)))

sai : [(1, 4), (4, 1)]
đúng: [(1, 4), (2, 1)]


**Vì sao sai:** `f = x² + y` không đối xứng: `f(1, 4) = 5` nhưng `f(4, 1) = 17`. Ta không được giả định gì về công thức ngoài tính đơn điệu. **Sửa:** xét đủ cả hai chiều như hai con trỏ.

## 14. Mẹo & nhận dạng pattern

- "Hàm ẩn / ma trận tăng theo hàng và theo cột, tìm mọi ô bằng `z`" → **hai con trỏ từ góc trên phải (hoặc dưới trái)**, `O(m + n)`.
- Một biến cố định, dãy theo biến kia tăng → **binary search**.
- Bài interactive: tự viết **mock** đếm số lần gọi để so sánh hiệu quả.
- Giá trị **nguyên** và tăng **ngặt** → mỗi bước tăng ít nhất 1 → suy ra chặn trên từ giá trị đích (như `x, y ≤ z`).
- Các bài cùng khung: 240 Search a 2D Matrix II, 167 Two Sum II (mảng đã sắp xếp), 378 Kth Smallest in Sorted Matrix, 1351 Count Negative Numbers in a Sorted Matrix.

## 15. Biến thể

1. **Đếm số nghiệm** thay vì liệt kê → giữ nguyên hai con trỏ, tăng biến đếm.
2. **Đếm số cặp có `f(x, y) ≤ z`** → hai con trỏ: với mỗi `x`, `y` lớn nhất thoả mãn đóng góp `y` cặp (giống đếm trong ma trận sắp xếp, nền tảng của bài k-th smallest).
3. **Two Sum II (LeetCode 167):** `f(i, j) = a[i] + a[j]` với `a` tăng → cùng kỹ thuật hai con trỏ.
4. **Hàm giảm theo một biến** (`f` tăng theo `x`, giảm theo `y`) → đổi góc xuất phát thành `(1, 1)`.
5. **Ba biến** `f(x, y, w)` tăng ngặt → cố định `x`, hai con trỏ trên `(y, w)`: `O(X · (Y + W))`.

Demo biến thể 2 (đếm cặp `f ≤ z`):

In [23]:
def count_leq(cf, z):
    total = 0
    x, y = 1, z
    while x <= z and y >= 1:
        if cf.f(x, y) <= z:
            total += y          # cả hàng x từ 1..y đều <= z
            x += 1
        else:
            y -= 1
    return total


for fid in (1, 2, 5):
    brute = sum(1 for x in range(1, 101) for y in range(1, 101) if FUNCS[fid](x, y) <= 20)
    print(fid, count_leq(CustomFunction(FUNCS[fid]), 20), brute)
    assert count_leq(CustomFunction(FUNCS[fid]), 20) == brute

1 190 190
2 66 66
5 13 13


## 16. So sánh các cách

| Cách | Ý tưởng | Time (lời gọi f) | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn (+ cắt) | Thử mọi `(x, y)` | `10⁶` (hoặc `O(X + z²)`) | `O(1)` | Làm tham chiếu kiểm thử |
| 2. Nhị phân theo `y` | Mỗi `x` tìm nhị phân | `O(X log Y)` | `O(1)` | Khi chỉ một chiều đơn điệu được khai thác |
| 3. Hai con trỏ | Góc trên phải, bỏ hàng/cột | `O(X + Y)` | `O(1)` | Lời giải chuẩn, dễ nhớ |
| 4. Hai con trỏ + chặn `z` | Như 3, miền `[1, z]` | `O(z)` | `O(1)` | Tốt nhất cho đề này |
| 5. Nhị phân co cận | Nhị phân, cận `y` giảm dần | `O(z log z)` | `O(1)` | Khi một chiều rất dài |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `f(x, y) = x * x + y`, `z = 10`. Các nghiệm là gì?

**Bài 2 (tính tay):** với `f = x + y`, `z = 4`, hai con trỏ (Cách 4) gọi `f` bao nhiêu lần?

**Bài 3 (điền TODO):** viết `count_solutions(cf, z)` trả về **số** cặp nghiệm bằng hai con trỏ.

**Bài 4 (tự viết lại):** viết `find_bottom_left(cf, z)` dùng hai con trỏ xuất phát từ góc **dưới trái** `(x = z, y = 1)`.

**Bài 5 (mini challenge):** cho mảng tăng ngặt `a` và `b`, đếm số cặp `(i, j)` có `a[i] + b[j] == t` trong `O(len(a) + len(b))`.

In [24]:
def count_solutions(cf, z):
    # TODO: hai con trỏ, trả về số nghiệm
    raise NotImplementedError


def find_bottom_left(cf, z):
    # TODO: xuất phát (x=z, y=1); f<z -> y+=1 ; f>z -> x-=1
    raise NotImplementedError


def count_pair_sum(a, b, t):
    # TODO: a, b tăng ngặt
    raise NotImplementedError


# Test gợi ý:
# assert count_solutions(CustomFunction(FUNCS[1]), 5) == 4
# assert sorted(find_bottom_left(CustomFunction(FUNCS[2]), 5)) == [[1, 5], [5, 1]]
# assert count_pair_sum([1, 2, 3], [2, 3, 4], 5) == 3

## 18. Gợi ý 3 mức

**Bài 1:**
- Mức 1: `x² ≤ 9` nên `x ∈ {1, 2, 3}`.
- Mức 2: `y = 10 - x²`.
- Mức 3: kiểm tra `y ≥ 1`.

**Bài 2:**
- Mức 1: bắt đầu `(1, 4)`.
- Mức 2: viết bảng như mục 8.
- Mức 3: dừng khi `x > 4` hoặc `y < 1`.

**Bài 3–5:**
- Mức 1: chép khung hai con trỏ ở mục 9.
- Mức 2: Bài 4: ở góc dưới trái, tăng `y` làm `f` tăng, giảm `x` làm `f` giảm.
- Mức 3: Bài 5: `i` từ đầu `a`, `j` từ cuối `b`; tổng nhỏ → `i += 1`, lớn → `j -= 1`, bằng → đếm và dịch cả hai (vì tăng ngặt).

## 19. Đáp án tham khảo

**Bài 1:** `x = 1 → y = 9`, `x = 2 → y = 6`, `x = 3 → y = 1` → `[[1,9],[2,6],[3,1]]`.

**Bài 2:** `(1,4)=5>4 → y=3`, `(1,3)=4 ghi`, `(2,2)=4 ghi`, `(3,1)=4 ghi`, `x=4,y=0` dừng → **4** lần gọi.

In [25]:
assert norm(Solution().findSolution(CustomFunction(FUNCS[3]), 10)) == [(1, 9), (2, 6), (3, 1)]
cf = CustomFunction(FUNCS[1])
SolutionTwoPointersBounded().findSolution(cf, 4)
assert cf.calls == 4
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [26]:
def count_solutions(cf, z):
    cnt = 0
    x, y = 1, z
    while x <= z and y >= 1:
        v = cf.f(x, y)
        if v < z:
            x += 1
        elif v > z:
            y -= 1
        else:
            cnt += 1
            x += 1
            y -= 1
    return cnt


def find_bottom_left(cf, z):
    ans = []
    x, y = z, 1
    while x >= 1 and y <= z:
        v = cf.f(x, y)
        if v < z:
            y += 1
        elif v > z:
            x -= 1
        else:
            ans.append([x, y])
            x -= 1
            y += 1
    return ans


def count_pair_sum(a, b, t):
    i, j, cnt = 0, len(b) - 1, 0
    while i < len(a) and j >= 0:
        s = a[i] + b[j]
        if s < t:
            i += 1
        elif s > t:
            j -= 1
        else:
            cnt += 1
            i += 1
            j -= 1
    return cnt


assert count_solutions(CustomFunction(FUNCS[1]), 5) == 4
assert sorted(find_bottom_left(CustomFunction(FUNCS[2]), 5)) == [[1, 5], [5, 1]]
assert count_pair_sum([1, 2, 3], [2, 3, 4], 5) == 3
rng = random.Random(5)
for _ in range(200):
    func = random_monotone(rng)
    z = rng.randint(1, 100)
    base = run_all(func, z)
    assert count_solutions(CustomFunction(func), z) == len(base)
    assert norm(find_bottom_left(CustomFunction(func), z)) == base
    a = sorted(rng.sample(range(50), rng.randint(0, 10)))
    b = sorted(rng.sample(range(50), rng.randint(0, 10)))
    t = rng.randint(0, 100)
    assert count_pair_sum(a, b, t) == sum(1 for p in a for q in b if p + q == t)
print("Bài 3, 4, 5 OK")

Bài 3, 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (hai con trỏ trên hàm/ma trận tăng theo hai chiều):**

```python
class Solution:
    def findSolution(self, customfunction: 'CustomFunction', z: int) -> List[List[int]]:
        ans = []
        x, y = 1, min(z, 1000)
        while x <= 1000 and y >= 1:
            v = customfunction.f(x, y)
            if v < z:
                x += 1
            elif v > z:
                y -= 1
            else:
                ans.append([x, y])
                x += 1
                y -= 1
        return ans
```

**Checklist trước khi submit:**

- [ ] Xuất phát từ góc `(x nhỏ, y lớn)` — hai nước đi có tác dụng ngược nhau.
- [ ] `f < z` → `x += 1`; `f > z` → `y -= 1`; `f == z` → ghi rồi **dịch cả hai**.
- [ ] Biên bao gồm cả `1` và cận trên (`≤ 1000` hoặc `≤ z`).
- [ ] Không giả định công thức (đối xứng, tuyến tính...).
- [ ] Trả về list các list `[x, y]`, thứ tự bất kỳ.